# Módulo 05: Aceleración Columnar con Apache Arrow y Vectorized Pandas UDF

## 1. El Cuello de Botella de las UDFs Estándar de Python

En PySpark clásico, cuando una transformación no se puede resolver con funciones nativas de `pyspark.sql.functions`, los ingenieros recurrían a UDFs estándar (`@udf`). Esto introduce un problema severo de rendimiento:

```text
[JVM (Executors)]
│ (Serialización fila por fila con Pickle)
▼
[Subproceso Python Worker] ──> Procesa 1 fila ──> Serializa respuesta
│
▼
[JVM (Executors)]
```

* **Costo de serialización:** La JVM debe empaquetar cada fila individual en formato Pickle, enviarla a través de sockets IPC al worker de Python y reconstruirla como un objeto nativo de Python.
* **Sin optimización SIMD:** Al procesar elemento a elemento, el hardware no puede utilizar vectorización ni operaciones masivas en memoria.

---

## 2. La Solución Moderna: Apache Arrow y Pandas UDFs

**Apache Arrow** es un formato estándar de memoria columnar abierta. En lugar de transferir filas independientes serializadas:
1. La JVM y el proceso de Python comparten los bloques de datos directamente en formato Arrow en memoria.
2. Spark transfiere **lotes columnares completos** (*Batches*) al subproceso de Python.
3. El worker de Python recibe los datos como objetos vectorizados de `pandas.Series` o `pandas.DataFrame`, ejecutando operaciones escritas en C/NumPy con paralelismo a nivel de instrucciones (SIMD).

In [1]:
import os
import sys

# 1. Forzar a Spark a utilizar el Python del entorno Conda activo (spark_env)
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

from pyspark.sql import SparkSession
import pandas as pd
from pyspark.sql.functions import pandas_udf
import pyspark.sql.functions as F

# 2. Inicialización canónica con soporte Arrow
spark = (
    SparkSession.builder
    .appName("05_Arrow_y_Pandas_UDF")
    .master("local[*]")
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")
    .config("spark.sql.execution.arrow.maxRecordsPerBatch", "10000")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .getOrCreate()
)

print(f"Intérprete Python vinculado: {sys.executable}")
print(f"Apache Arrow activo: {spark.conf.get('spark.sql.execution.arrow.pyspark.enabled')}")

Intérprete Python vinculado: c:\Users\Ruben\anaconda3\envs\spark_env\python.exe
Apache Arrow activo: true


## 3. Tipos de Pandas UDFs (Evolución de la API)

Desde PySpark 3.0+, las Pandas UDFs se definen con anotaciones de tipos de Python estándar (`typing`). Existen tres patrones principales:

1. **Series to Series (`pd.Series -> pd.Series`):** Mismo número de entradas que de salidas (transformaciones columnares).
2. **Series to Scalar (`pd.Series -> float/int`):** Múltiples entradas consolidadas en un valor escalar (agregaciones sobre grupos o ventanas).
3. **Iterator of Series to Iterator of Series:** Para transformaciones que requieren inicializaciones costosas (como cargar un modelo de Machine Learning una sola vez por partición en lugar de hacerlo por lote).

In [2]:
import sys
import os
sys.path.append(os.path.abspath(".."))
from src.olympics_pipeline import DEPORTISTAS_SCHEMA

# Carga de datos
df_deportistas = (
    spark.read
    .schema(DEPORTISTAS_SCHEMA)
    .option("header", "true")
    .csv("../data/raw/deportista.csv")
)

# 1. Definición de Pandas UDF (Series to Series)
# Convierte altura de centímetros a pulgadas (1 cm = 0.393701 pulgadas)
@pandas_udf("double")
def cm_a_pulgadas(columna_cm: pd.Series) -> pd.Series:
    # La operación se ejecuta vectorizada en C a través de Pandas / NumPy
    return (columna_cm * 0.393701).round(2)

# 2. Aplicación sobre el DataFrame distribuido
df_con_pulgadas = df_deportistas.withColumn(
    "altura_pulgadas",
    cm_a_pulgadas(F.col("altura"))
)

df_con_pulgadas.select("deportista_id", "nombre", "altura", "altura_pulgadas").show(5)

+-------------+--------------------+------+---------------+
|deportista_id|              nombre|altura|altura_pulgadas|
+-------------+--------------------+------+---------------+
|            1|           A Dijiang| 180.0|          70.87|
|            2|            A Lamusi| 170.0|          66.93|
|            3|      Gunnar Nielsen| 185.0|          72.83|
|            4|Edgar Lindenau Aabye| 182.0|          71.65|
|            5|Christine Jacoba ...| 185.0|          72.83|
+-------------+--------------------+------+---------------+
only showing top 5 rows



## 4. Inspección del Plan Físico para Operaciones Arrow

Al examinar el plan de ejecución con `.explain()`, Spark registra explícitamente el operador columnar `ArrowEvalPython` en lugar del operador legado `BatchEvalPython`.

In [3]:
# Inspeccionar el operador ArrowEvalPython en el plan físico
df_con_pulgadas.select("nombre", "altura_pulgadas").explain(mode="formatted")

== Physical Plan ==
* Project (3)
+- ArrowEvalPython (2)
   +- Scan csv  (1)


(1) Scan csv 
Output [2]: [nombre#1, altura#4]
Batched: false
Location: InMemoryFileIndex [file:/c:/Users/Ruben/Desktop/Ciencia de datos/curso_spark/data/raw/deportista.csv]
ReadSchema: struct<nombre:string,altura:double>

(2) ArrowEvalPython
Input [2]: [nombre#1, altura#4]
Arguments: [cm_a_pulgadas(altura#4)#14], [pythonUDF0#52], 200

(3) Project [codegen id : 1]
Output [2]: [nombre#1, pythonUDF0#52 AS altura_pulgadas#15]
Input [3]: [nombre#1, altura#4, pythonUDF0#52]




## 5. Reto Práctico: Escalado Vectorizado Min-Max

### Instrucciones del Ejercicio:
1. Diseña una función con `@pandas_udf("double")` llamada `escalar_a_gramos` que reciba una columna de peso en kilogramos (`pd.Series`) y retorne el peso convertido en **gramos** (`peso * 1000.0`).
2. Aplica la función sobre `df_deportistas` creando la columna `"peso_gramos"`.
3. Filtra el registro correspondiente a `"Cornelia Aalten"`.
4. Extrae el valor de `"peso_gramos"` y valida el resultado con la aserción automática.

In [4]:
# 1. Definición de la función vectorizada
@pandas_udf("double")
def escalar_a_gramos(peso_kg: pd.Series) -> pd.Series:
    return peso_kg * 1000.0

# 2. Aplicación en el DataFrame
df_transformado = df_deportistas.withColumn(
    "peso_gramos",
    escalar_a_gramos(F.col("peso"))
)

df_transformado.select("nombre", "peso", "peso_gramos").show(5)

# 3. Extracción del valor para Cornelia Aalten (peso original = 55.0 kg)
peso_cornelia = (
    df_transformado
    .filter(F.col("nombre") == "Cornelia Aalten")
    .select("peso_gramos")
    .collect()[0]["peso_gramos"]
)

print(f"Peso de Cornelia Aalten en gramos: {peso_cornelia}")

# 4. Aserción de validación formal
assert peso_cornelia == 55000.0, f"Error: Se esperaba 55000.0 pero se obtuvo {peso_cornelia}"

print("¡Aserción aprobada! Cuaderno 05 completado exitosamente.")

+--------------------+----+-----------+
|              nombre|peso|peso_gramos|
+--------------------+----+-----------+
|           A Dijiang|80.0|    80000.0|
|            A Lamusi|60.0|    60000.0|
|      Gunnar Nielsen|82.0|    82000.0|
|Edgar Lindenau Aabye|81.0|    81000.0|
|Christine Jacoba ...|72.0|    72000.0|
+--------------------+----+-----------+
only showing top 5 rows

Peso de Cornelia Aalten en gramos: 55000.0
¡Aserción aprobada! Cuaderno 05 completado exitosamente.
